In [ ]:
# Cell 1 — Install
!pip install -q transformers accelerate flask pyngrok torch sentencepiece

In [ ]:
# Cell 2 — HuggingFace auth (need access to meta-llama/Meta-Llama-3-8B)
from huggingface_hub import login
login("Your_huggingface_token")   # or use Kaggle secrets

# Cell 3 — ngrok token for Instance 2 (must be a separate ngrok account)
import os
os.environ["NGROK_TOKEN"] = "Your_ngrok_token2" #microsoft-edge-avinash

In [ ]:
"""
╔══════════════════════════════════════════════════════════╗
║         LLAMA 3 8B — PIPELINE STAGE 1                   ║
║         Kaggle Instance 1                                ║
║         Layers: 11–21                                    ║
║         Role: Recv hidden states → Forward → Relay      ║
╚══════════════════════════════════════════════════════════╝

SETUP (run these cells first in Kaggle):
  !pip install -q transformers accelerate flask pyngrok torch requests safetensors huggingface_hub
  !huggingface-cli login  # paste your HF token when prompted

IMPORTANT: Boot THIS notebook SECOND (after Stage 2 is running).
  1. Copy Stage 2's ngrok URL → paste below as STAGE2_URL.
  2. Run this notebook.
  3. Copy the ngrok URL it prints → paste into stage0_server.py as STAGE1_URL.
"""

# ── Imports ──────────────────────────────────────────────────────────────────
import os, io, time, copy, requests
import torch
import numpy as np
from flask import Flask, request, jsonify
from transformers import AutoConfig, DynamicCache
from transformers.models.llama.modeling_llama import LlamaDecoderLayer
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
import torch.nn as nn
from pyngrok import ngrok

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID          = "meta-llama/Meta-Llama-3-8B-Instruct"
SPLIT_LAYER_START = 11          # Stage 1 owns layers 11..21 (inclusive)
SPLIT_LAYER_END   = 22          # exclusive upper bound
STAGE2_URL        = None        # Set after Stage 2 boots (see below)
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE             = torch.float16

print(f"[Stage 1] Device: {DEVICE}  |  dtype: {DTYPE}")

    # ── Model Loading ─────────────────────────────────────────────────────────────
class LlamaStage1(nn.Module):
    """
    Holds: rotary_emb + layers[11:22]  — no embed, no norm, no lm_head
    Input : hidden states  (batch, 1, hidden_size)  — from Stage 0
    Output: hidden states  (batch, 1, hidden_size)  — passed to Stage 2
    """
    def __init__(self, config, split_layer_start=SPLIT_LAYER_START, split_layer_end=SPLIT_LAYER_END):
        super().__init__()
        config._attn_implementation = "eager"
        self.config = config
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(config, layer_idx=i)
            for i in range(split_layer_start, split_layer_end)
        ])
        from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
        self.rotary_emb = LlamaRotaryEmbedding(config=config)  # ✅ build once
    
    def forward(self, hidden_states, past_key_values=None, past_len_override=0):
        seq_len  = hidden_states.shape[1]
        past_len = past_len_override
    
        position_ids   = torch.arange(past_len, past_len + seq_len,
                                      device=hidden_states.device).unsqueeze(0)
        cache_position = torch.arange(past_len, past_len + seq_len,
                                      device=hidden_states.device)
    
        if past_key_values is None:
            past_key_values = DynamicCache()
    
        cos, sin = self.rotary_emb(hidden_states, position_ids)
        position_embeddings = (cos, sin)
    
        hidden = hidden_states
        for layer in self.layers:
            out = layer(
                hidden,
                attention_mask=None,
                position_ids=position_ids,
                past_key_values=past_key_values,
                use_cache=True,              # ✅ must cache for autoregressive decode
                cache_position=cache_position,
                position_embeddings=position_embeddings,
            )
            hidden = out if isinstance(out, torch.Tensor) else out[0]
            if hidden.dim() == 2:
                hidden = hidden.unsqueeze(1)
    
        if seq_len > 1:
            return hidden, past_key_values
        else:
            return hidden[:, -1:, :], past_key_values


def load_stage1_model():
    print("[Stage 1] Loading config...")
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)

    print(f"[Stage 1] Building model "
          f"(layers {SPLIT_LAYER_START}–{SPLIT_LAYER_END - 1})...")
    model = LlamaStage1(config)

    print("[Stage 1] Loading pretrained weights — targeted load, layers "
          f"{SPLIT_LAYER_START}–{SPLIT_LAYER_END - 1} only...")
    # ── Targeted shard loader ─────────────────────────────────────────────────
    # Instead of loading the full 16 GB model into CPU RAM and then discarding
    # most of it, we read only the weight keys that belong to our layers directly
    # from the safetensors shards. This cuts CPU RAM usage from ~16 GB → ~5 GB
    # and reduces download time proportionally.
    from huggingface_hub import snapshot_download
    from safetensors import safe_open

    # Download only the safetensors shards (no PyTorch bin files needed)
    model_dir = snapshot_download(
        MODEL_ID,
        ignore_patterns=["*.bin", "*.bin.index.json"],   # skip old-format weights
    )

    # Build the set of parameter name prefixes we actually need
    # HF stores them as "model.layers.{i}...." inside the shards
    our_prefixes = tuple(
        f"model.layers.{i}." for i in range(SPLIT_LAYER_START, SPLIT_LAYER_END)
    )

    # Walk every shard and pull out only our keys
    import glob, os as _os
    shard_files = sorted(glob.glob(_os.path.join(model_dir, "*.safetensors")))
    state_dict  = {}
    for shard_path in shard_files:
        with safe_open(shard_path, framework="pt", device="cpu") as f:
            for key in f.keys():
                if key.startswith(our_prefixes):
                    state_dict[key] = f.get_tensor(key).to(DTYPE)

    # Re-key from "model.layers.{i}.X" → "layers.{local_i}.X" to match our module
    remapped = {}
    for key, tensor in state_dict.items():
        # key looks like "model.layers.11.self_attn.q_proj.weight"
        parts      = key.split(".")            # ["model", "layers", "11", ...]
        global_idx = int(parts[2])
        local_idx  = global_idx - SPLIT_LAYER_START
        new_key    = "layers." + str(local_idx) + "." + ".".join(parts[3:])
        remapped[new_key] = tensor

    missing, unexpected = model.load_state_dict(remapped, strict=True)
    if missing:
        print(f"[Stage 1] ⚠️  Missing keys: {missing[:5]}{'...' if len(missing)>5 else ''}")
    # ─────────────────────────────────────────────────────────────────────────

    import gc; gc.collect()

    model = model.to(DTYPE).to(DEVICE).eval()
    print(f"[Stage 1] Model loaded. VRAM used: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return model, config


# ── Tensor helpers ────────────────────────────────────────────────────────────
def tensor_to_bytes(t: torch.Tensor) -> bytes:
    """Serialize tensor to raw bytes (numpy framing)."""
    arr = t.cpu().to(torch.float16).numpy()
    buf = io.BytesIO()
    np.save(buf, arr)
    return buf.getvalue()

def bytes_to_tensor(b: bytes) -> torch.Tensor:
    buf = io.BytesIO(b)
    arr = np.load(buf)
    return torch.from_numpy(arr).to(DTYPE).to(DEVICE)

# ── KV-Cache Store ────────────────────────────────────────────────────────────
# We keep a simple in-memory dict: session_id -> {"pkv_1": [...]}
# For research/single-session use this is fine.
SESSIONS = {}

# ── Flask App ─────────────────────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"stage": 1, "status": "ok",
                    "vram_gb": torch.cuda.memory_allocated() / 1e9})

@app.route("/forward", methods=["POST"])
def forward():
    session_id  = request.headers.get("X-Session-Id", "default")
    step        = int(request.headers.get("X-Step", 0))
    temperature = float(request.headers.get("X-Temperature", 0.8))
    top_p       = float(request.headers.get("X-Top-P", 0.95))
    past_len    = int(request.headers.get("X-Past-Len", 0))
    seq_len     = int(request.headers.get("X-Seq-Len", 1))

    # print(f"[Stage 1] step={step} past_len={past_len} seq_len={seq_len}")

    hidden = bytes_to_tensor(request.data)  # (1, seq_len, hidden_size)

    # ✅ Restore KV cache for this session
    pkv_1 = SESSIONS.get(session_id, {}).get("pkv_1", None)

    with torch.no_grad():
        hidden_out, pkv_1 = model(hidden, past_key_values=pkv_1,
                                  past_len_override=past_len)

    # ✅ Save updated KV cache
    SESSIONS[session_id] = {"pkv_1": pkv_1}

    payload_bytes = tensor_to_bytes(hidden_out)  # always (1, 1, hidden_size)
    resp = requests.post(
        f"{STAGE2_URL}/forward",
        data=payload_bytes,
        headers={
            "Content-Type": "application/octet-stream",
            "X-Session-Id":  session_id,
            "X-Step":        str(step),
            "X-Temperature": str(temperature),
            "X-Top-P":       str(top_p),
            "X-Past-Len":    str(past_len + seq_len - 1),  # position of last token
            "X-Seq-Len":     "1",                          # always 1 after stage 1
        },
        timeout=60,
    )
    resp.raise_for_status()
    return jsonify(resp.json())

@app.route("/reset_session", methods=["POST"])
def reset_session():
    """Clear KV cache for a session (call before a new conversation)."""
    session_id = request.json.get("session_id", "default")
    SESSIONS.pop(session_id, None)
    return jsonify({"cleared": session_id})


# ── Boot Sequence ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load model
    model, config = load_stage1_model()

    # 2. Set STAGE2_URL  ← YOU MUST FILL THIS IN after Stage 2 is running
    #    After running stage2_server.py in the other Kaggle notebook,
    #    copy the ngrok URL it prints and paste it here.
    STAGE2_URL = "https://800c-136-115-66-183.ngrok-free.app"
    print(f"[Stage 1] Will relay hidden states to Stage 2 at: {STAGE2_URL}")

    # 3. Start ngrok tunnel for this server (so Stage 0 can reach Stage 1)
    ngrok_token = os.environ.get("NGROK_TOKEN", "")   # set via Kaggle secrets
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(5001)
    print(f"\n{'='*60}")
    print(f"[Stage 1] PUBLIC URL: {tunnel.public_url}")
    print(f"  → Copy this URL into stage0_server.py as STAGE1_URL")
    print(f"  → e.g.  STAGE1_URL = \"{tunnel.public_url}\"")
    print(f"{'='*60}\n")

    # 4. Run Flask
    app.run(host="0.0.0.0", port=5001, threaded=False)